In [0]:
import requests, json
from datetime import datetime, timezone
from pyspark.sql import functions as F

CATALOG = "workspace_practica_1"
BASE = "https://www.datos.gov.co/resource/jbjy-vk9h.json"

Análisis de columnas.
Se busca asegurar que se encuentren las columnas: departamento, nombre_entidad, valor_del_contrato, fecha_de_firma y modalidad_de_contratacion

In [0]:
muestra = requests.get(BASE, params={"$limit": 3}, timeout=60).json()
print(f"Número de columnas: {len(muestra[0])}")
print(sorted(muestra[0].keys()))

Análisis de volúmen de datos que se pueden obtener

In [0]:
def diagnostico(dataset_id):
    url = f"https://www.datos.gov.co/resource/{dataset_id}.json"
    print(f"\n===== {dataset_id} =====")

    total = requests.get(url, params={"$select": "count(*) as total"}, timeout=120).json()
    print("Total de registros:", total)

    rango = requests.get(url, params={
        "$select": "min(fecha_de_firma) as minima, max(fecha_de_firma) as maxima"
    }, timeout=120).json()
    print("Rango de fechas de firma:", rango)

    nulos = requests.get(url, params={
        "$select": "count(*) as sin_fecha", "$where": "fecha_de_firma IS NULL"
    }, timeout=120).json()
    print("Registros sin fecha de firma:", nulos)

    por_anio = requests.get(url, params={
        "$select": "date_extract_y(fecha_de_firma) as anio, count(*) as total",
        "$group": "anio", "$order": "anio"
    }, timeout=180).json()
    print("Contratos por año:")
    for fila in por_anio:
        print(f"   {fila.get('anio')}: {int(fila['total']):,}")

diagnostico("jbjy-vk9h")   # SECOP II - Contratos Electrónicos
diagnostico("p8vk-huva")   # SECOP II - Contratos Electrónicos - ACTIVOS

La cantidad dice ser 1000 registros. Lo que es extraño debido a que solo en un departamento en un año pueden haber miles de contratos
Hipótesis: Se está limitando la cantidad de registros que se pueden obtener desde la API sin tener un token de acceso.

A continuación se realiza la prueba de esta hipótesis.

In [0]:
url_trm = "https://www.datos.gov.co/resource/32sa-8pi3.json"
print(requests.get(url_trm, params={"$select": "count(*) as total"}, timeout=60).json())

8.352 registros, más de mil. Eso descarta que el portal esté limitando todas las consultas anónimas. El problema es específico del dataset de SECOP II.

Probablemente ese dataset exige autenticación para acceder al conjunto completo, y sin token solo expone una muestra de 1.000 filas. 
Sin embargo podemos verificar si el dataset fue reemplazado o se encuentra abandonado

In [0]:
meta = requests.get("https://www.datos.gov.co/api/views/jbjy-vk9h.json", timeout=60).json()
from datetime import datetime, timezone
print("Nombre:", meta.get("name"))
print("Última actualización de filas:",
      datetime.fromtimestamp(meta.get("rowsUpdatedAt", 0), tz=timezone.utc))
print("Descripción:", (meta.get("description") or "")[:500])

El dataset se actualizó hoy mismo y la descripción dice que contiene todos los contratos de SECOP II desde su lanzamiento. No es un conjunto reducido ni abandonado.

Entonces la explicación más probable es la primera: el portal limita a 1.000 filas las consultas anónimas sobre este dataset

**Lo recomendado es crear una API desde datos.gov.co y reiniciar el proceso de consumo de datos**


In [0]:
import os
HEADERS = {"X-App-Token": os.environ.get("SOCRATA_APP_TOKEN", "")}
print("Token cargado:", bool(HEADERS["X-App-Token"]))

print(requests.get(BASE, params={"$select": "count(*) as total"},
                   headers=HEADERS, timeout=120).json())

La salida sigue limitada a 1000. Por lo que este dataset no es muy útil.

In [0]:
candidatos = {
    "vwyg-ip7i": "SECOP II - Contratos Electrónicos del Departamento de Sucre",
    "ay65-guja": "Vista SECOP II - Contratos Electrónicos",
    "p6dx-8zbt": "SECOP II - Procesos de Contratación",
    "t3dm-3p82": "SECOP II - Contratos Electrónicos - PYMES",
}

for ds_id, nombre in candidatos.items():
    url = f"https://www.datos.gov.co/resource/{ds_id}.json"
    try:
        total = requests.get(url, params={"$select": "count(*) as total"},
                             headers=HEADERS, timeout=120).json()
        muestra = requests.get(url, params={"$limit": 1}, headers=HEADERS, timeout=60).json()
        tiene_fecha = "fecha_de_firma" in muestra[0] if muestra else False
        print(f"{ds_id} | {nombre}\n   total: {total} | tiene fecha_de_firma: {tiene_fecha}\n")
    except Exception as e:
        print(f"{ds_id} | {nombre}\n   ERROR: {e}\n")

El dataset p6dx-8zbt sí expone el conjunto completo por API con 9,2 millones de registros.

Además es una muy buena fuente para la práctica, con una diferencia conceptual:

Contratos electrónicos: los contratos ya firmados.
Procesos de contratación: cada proceso de compra que publica una entidad, se haya adjudicado o no. Incluye el precio base con el que se abrió, cuántos proveedores participaron y cómo terminó.

Eso permite análisis como qué porcentaje de procesos se declaran desiertos, cuánta competencia hay por modalidad, o la diferencia entre el precio base y lo que finalmente se adjudicó.

In [0]:
BASE = "https://www.datos.gov.co/resource/p6dx-8zbt.json"

muestra = requests.get(BASE, params={"$limit": 1}, headers=HEADERS, timeout=60).json()
print(f"Número de columnas: {len(muestra[0])}\n")
for k in sorted(muestra[0].keys()):
    valor = str(muestra[0][k])[:60]
    print(f"{k:<45} {valor}")

# Luego de identificar las nuevas columnas podemos reiniciar el enfoque del proceso

In [0]:
import os, requests, json
from datetime import datetime, timezone
from pyspark.sql import functions as F

CATALOG = "workspace_practica_1"
BASE = "https://www.datos.gov.co/resource/p6dx-8zbt.json"
HEADERS = {"X-App-Token": os.environ.get("SOCRATA_APP_TOKEN", "")}

## Procesos por departamento

In [0]:
FILTRO_ANIO = ("fecha_de_publicacion_del between "
               "'2025-01-01T00:00:00' and '2025-12-31T23:59:59'")

conteo = requests.get(BASE, params={
    "$select": "departamento_entidad, count(*) as total",
    "$where": FILTRO_ANIO,
    "$group": "departamento_entidad",
    "$order": "total DESC"
}, headers=HEADERS, timeout=180).json()

for fila in conteo:
    print(f"{fila.get('departamento_entidad', 'SIN DATO'):<45} {int(fila['total']):>10,}")

In [0]:
DEPARTAMENTO = "Quindío"   # cámbialo por el que elegiste, escrito exactamente igual
WHERE = f"{FILTRO_ANIO} AND departamento_entidad = '{DEPARTAMENTO}'"
LOTE, MAX_FILAS = 10_000, 50_000

registros = []
for offset in range(0, MAX_FILAS, LOTE):
    r = requests.get(BASE, params={
        "$where": WHERE, "$limit": LOTE,
        "$offset": offset, "$order": ":id"
    }, headers=HEADERS, timeout=180)
    r.raise_for_status()
    lote = r.json()
    print(f"Offset {offset}: {len(lote)} registros")
    if not lote:
        break
    registros.extend(lote)

print(f"Total descargado: {len(registros):,}")

Descarga completa: 31.552 procesos del Quindío en 2025, justo en el rango ideal. 
Se puede observar además que el ciclo se detuvo solo cuando la API devolvió 0 registros; así funciona la paginación


**Guardar el archivo crudo en el volumen**

In [0]:
ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
ruta = f"/Volumes/{CATALOG}/secop_bronze/raw/procesos_{ts}.json"

with open(ruta, "w", encoding="utf-8") as f:
    for reg in registros:
        f.write(json.dumps(reg, ensure_ascii=False) + "\n")

print(f"Archivo guardado en: {ruta}")

## Creación tabla BRONZE

In [0]:
bronze = (spark.read.json(f"/Volumes/{CATALOG}/secop_bronze/raw/")
          .withColumn("_archivo_origen", F.col("_metadata.file_path"))
          .withColumn("_fecha_ingesta", F.current_timestamp()))

(bronze.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.secop_bronze.procesos"))

print(f"Filas en bronze: {spark.table(f'{CATALOG}.secop_bronze.procesos').count():,}")

In [0]:
# Revisar el resultado anterior

df = spark.table(f"{CATALOG}.secop_bronze.procesos")
df.printSchema()
display(df.limit(10))

Primera pregunta de calidad

Esta es nueva. Verifica si id_del_proceso es realmente único, porque de eso depende cómo diseñemos silver:

In [0]:
total = df.count()
unicos = df.select("id_del_proceso").distinct().count()
print(f"Filas: {total:,} | id_del_proceso únicos: {unicos:,} | repetidos: {total - unicos:,}")

display(df.groupBy("id_del_proceso").count()
          .filter("count > 1")
          .orderBy(F.desc("count"))
          .limit(10))